In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case)

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/01110
01110


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

In [7]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [8]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [9]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1':
    cntrl_vars_init = [1]
elif case[3] == '2':
    cntrl_vars_init = [0,1]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [10]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_' + case + '.pickle'
case_1 = case[0] + case[1] + '0' + case[3] + case[4]
final_file_1 = 'control_' + case_1 + '.pickle'

In [11]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

In [12]:
# get initial parameters and target states

i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
i_range_0 = range(0, len(exc),i_stepsize)
i_range_1 = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [13]:
# get uncontrolled cost

data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    
    if not type(bestControl_init[i]) == type(None):
        continue
        
    control0 = aln.getZeroControl()

    ##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)

    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    with open(init_file,'wb') as f:
        pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  1 , total integrated cost =  1214.8607754997602
RUN  2 , total integrated cost =  220.65900370108346
RUN  3 , total integrated cost =  106.80313385038696
RUN  4 , total integrated cost =  19.83441935245901
RUN  5 , total integrated cost =  18.332585410438945
RUN  6 , total integrated cost =  16.746012942355296
RUN  7 , total integrated cost =  15.660390012055887
RUN  8 , total integrated cost =  14.544178897041077
RUN  9 , total integrated cost =  13.625583827784723
RUN  10 , total integrated cost =  12.857445515000377
RUN  11 , total integrated cost =  11.808698580804997
RUN  12 , total integrated cost =  10.986213006985341
RUN  13 , total integrated cost =  10.487797512020435
RUN  14 , total integrated cost =  10.159351916074579
RUN  15 , total integrated cost =  10.095753023597073

ERROR:root:Problem in initial value trasfer


RUN  40 , total integrated cost =  9.002214134086712
Control only changes marginally.
RUN  41 , total integrated cost =  9.002214134086712
Improved over  41  iterations in  13.56566670909524  seconds by  99.84748230800858  percent.
Problem in initial value trasfer:  Vmean_exc -56.627610777868966 -56.627611258587805
weight =  6556.616396058647
set cost params:  1.0 6556.616396058647 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5846.098500984984
Gradient descend method:  None
RUN  1 , total integrated cost =  4164.690876893683
RUN  2 , total integrated cost =  4163.519935527499
RUN  3 , total integrated cost =  4163.197533759545
RUN  4 , total integrated cost =  4162.373999273211
RUN  5 , total integrated cost =  4124.0493465800455
RUN  6 , total integrated cost =  4118.178201246843
RUN  7 , total integrated cost =  4118.114155762954
RUN  8 , total integrated cost =  4118.10865963353
RUN  9 , total integrated cost =  4118.107927189125
RUN  10 , total integr

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  28 , total integrated cost =  4118.107686638131
Improved over  28  iterations in  0.6550724003463984  seconds by  29.55801743771015  percent.
Problem in initial value trasfer:  Vmean_exc -56.62687215656052 -56.62684083003213
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  705.4029734175052
RUN  2 , total integrated cost =  208.96595381043926
RUN  3 , total integrated cost =  100.33196111026527
RUN  4 , total integrated cost =  49.25978056695863
RUN  5 , total integrated cost =  46.7362846288914
RUN  6 , total integrated cost =  43.955874065773386
RUN  7 , total integrated cost =  40.08872858155644
RUN  8 , total integrated cost =  37.44356312274883
RUN  9 , total integrated cost =  35.783843214120644
RUN  10 , total integrated cost =  34.91521608842192
RUN  11 , t

...

RUN  1 , total integrated cost =  23532.63614302563
Control only changes marginally.
RUN  1 , total integrated cost =  23532.63614302563
Improved over  1  iterations in  0.09355634823441505  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -73.65794409599617 -73.6579441840707
-------  140 0.4500000000000001 0.9000000000000006
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10019.968518582271
Gradient descend method:  None
RUN  1 , total integrated cost =  10019.968518582271
Control only changes marginally.
RUN  1 , total integrated cost =  10019.968518582271
Improved over  1  iterations in  0.048979464918375015  seconds by  0.0  percent.
weight =  10.0
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10019.968518582271
Gradient descend method:  None
RUN  1 , total integrated cost =  10019.968518582271
Control only changes marginally.
RUN  1 , total integrat

ERROR:root:Problem in initial value trasfer


-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 10.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.05146687772
Gradient descend method:  None
RUN  1 , total integrated cost =  33290.047977386734
RUN  2 , total integrated cost =  33290.04797738673
RUN  3 , total integrated cost =  33290.04797738672
RUN  4 , total integrated cost =  33290.04797738672
Control only changes marginally.
RUN  4 , total integrated cost =  33290.04797738672
Improved over  4  iterations in  0.1753840632736683  seconds by  1.048208352472102e-05  percent.
Problem in initial value trasfer:  Vmean_exc -68.82423325167102 -68.8242522248443
weight =  10.000001048208462
set cost params:  1.0 10.000001048208462 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33290.047977387076
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  33290.047977387076
Control only changes marginally.
RUN  1 , total integrated cost =  33290.047977387076
Improved over  1  iterations in  0.06270043551921844  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -68.82423325167102 -68.8242522248443


In [15]:
"""
#plot initial guesses
for i in i_range:
    print("---------", i)
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
    plt.show()
"""

'\n#plot initial guesses\nfor i in i_range:\n    print("---------", i)\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n    plt.show()\n'

In [17]:
found_solution = []
no_solution = []
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break


    for i in i_range:
        print("------- ", i, exc[i], inh[i])        

        if np.abs(np.mean(bestState_init[i][0,0,-300:]) - target[i][0,0,-1]) < 0.1 * np.abs(
            np.mean(bestState_init[i][0,0,-100:]) - bestState_init[i][0,0,0]) and np.abs(
            np.mean(bestState_init[i][0,1,-300:]) - target[i][0,1,-1]) < 0.1 * np.abs(
            np.mean(bestState_init[i][0,1,-100:]) - bestState_init[i][0,1,0]) and np.amax(
            bestState_init[i][0,0,:]) < target[i][0,0,-1] + 1. and np.amax(
            bestState_init[i][0,1,:]) < target[i][0,1,-1]:
            # and np.amin(bestState_init[i][0,0,:]) > bestState_init[i][0,0,0] - 1.
            #and np.amin(bestState_init[i][0,1,:]) > bestState_init[i][0,1,0] - 1.:
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        if len(found_solution) == 0:
            continue
            
        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(init_file,'wb') as f:
            pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                         costnode_init, weights_init], f)

------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
-------  15 0.4500000000000001 0.4500000000000002
found solution for  15
-------  20 0.4500000000000001 0.4750000000000002
found solution for  20
-------  25 0.4250000000000001 0.5000000000000002
found solution for  25
-------  30 0.4250000000000001 0.5250000000000002
found solution for  30
-------  35 0.5500000000000003 0.5250000000000002
found solution for  35
-------  40 0.5250000000000001 0.5500000000000003
found solution for  40
-------  45 0.5000000000000002 0.5750000000000003
found solution for  45
-------  50 0.47500000000000014 0.6000000000000003
found solution for  50
-------  55 0.4250000000000001 0.62500

ERROR:root:Problem in initial value trasfer


RUN  30 , total integrated cost =  5.892423159856166
Control only changes marginally.
RUN  30 , total integrated cost =  5.892423159856166
Improved over  30  iterations in  0.7260870151221752  seconds by  79.65078617767269  percent.
Problem in initial value trasfer:  Vmean_exc -56.62762095165168 -56.62762091109556
weight =  10016.942638217553
set cost params:  1.0 10016.942638217553 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5809.661976458566
Gradient descend method:  None
RUN  1 , total integrated cost =  5171.026066057029
RUN  2 , total integrated cost =  5165.804306176467
RUN  3 , total integrated cost =  5164.376112857751
RUN  4 , total integrated cost =  5155.907038081823
RUN  5 , total integrated cost =  5151.995193602638
RUN  6 , total integrated cost =  5151.053654030334
RUN  7 , total integrated cost =  5137.866379741302
RUN  8 , total integrated cost =  5129.939053589568
RUN  9 , total integrated cost =  5129.428003709825
RUN  10 , total integ

ERROR:root:Problem in initial value trasfer


RUN  15 , total integrated cost =  5041.33164402058
RUN  16 , total integrated cost =  5041.331644020578
State only changes marginally.
RUN  17 , total integrated cost =  5041.331644020578
Control only changes marginally.
RUN  17 , total integrated cost =  5041.331644020578
Improved over  17  iterations in  0.48188032023608685  seconds by  13.225043652304606  percent.
Problem in initial value trasfer:  Vmean_exc -56.626374677823655 -56.62638459266206
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000

In [18]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print("------------------------------------------------")
    print('-------------------------', counter)
    
    if counter > 20:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        
        if conv_init[i] == [True, True]:
            continue
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
                       
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(init_file,'wb') as f:
            pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                         costnode_init, weights_init], f)
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
    counter += 1

------------------------------------------------
------------------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
weight =  11726.86702102814
set cost params:  1.0 11726.86702102814 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5887.122890522112
Gradient descend method:  None
RUN  1 , total integrated cost =  5886.155505359946
RUN  2 , total integrated cost =  5886.155503640004
RUN  3 , total integrated cost =  5886.155503639963
RUN  4 , total integrated cost =  5886.15550

ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  5886.155503639955
RUN  7 , total integrated cost =  5886.155503639952
State only changes marginally.
RUN  8 , total integrated cost =  5886.155503639952
Control only changes marginally.
RUN  8 , total integrated cost =  5886.155503639952
Improved over  8  iterations in  0.29258088022470474  seconds by  0.0164322522249023  percent.
Problem in initial value trasfer:  Vmean_exc -56.62628971199804 -56.626300333526885
no convergence
-------  5 0.4000000000000001 0.40000000000000013
weight =  1761.6196862842191
set cost params:  1.0 1761.6196862842191 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5090.869495380112
Gradient descend method:  None
RUN  1 , total integrated cost =  5090.8012207974325
RUN  2 , total integrated cost =  5090.799585564993
RUN  3 , total integrated cost =  5090.799498087841
RUN  4 , total integrated cost =  5090.799491826206
RUN  5 , total integrated cost =  5090.799491291518
RUN  6 , total integrated co

ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  5090.799491253452
RUN  8 , total integrated cost =  5090.799491253238
RUN  9 , total integrated cost =  5090.799491253211
RUN  10 , total integrated cost =  5090.799491253209
RUN  11 , total integrated cost =  5090.799491253206
RUN  12 , total integrated cost =  5090.799491253197
RUN  13 , total integrated cost =  5090.799491253196
RUN  14 , total integrated cost =  5090.799491253196
Control only changes marginally.
RUN  14 , total integrated cost =  5090.799491253196
Improved over  14  iterations in  0.38258364237844944  seconds by  0.0013750917594705925  percent.
Problem in initial value trasfer:  Vmean_exc -56.625406188469384 -56.6253877314189
no convergence
-------  10 0.4250000000000001 0.42500000000000016
weight =  2813.5185861886544
set cost params:  1.0 2813.5185861886544 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9074.309839288742
Gradient descend method:  None
RUN  1 , total integrated cost =  9072.00948418388

...

converged for  40
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
weight =  2570.2053050286795
set cost params:  1.0 2570.2053050286795 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  29784.051645940723
Gradient descend method:  None
RUN  1 , total integrated cost =  29784.051645940723
Control only changes marginally.
RUN  1 , total integrated cost =  29784.051645940723
Improved over  1  iterations in  0.08072448894381523  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70432988366847 -56.704329859248055


ERROR:root:Problem in initial value trasfer


converged for  60
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
weight =  823.3333979661207
set cost params:  1.0 823.3333979661207 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  24387.246117422153
Gradient descend method:  None
RUN  1 , total integrated cost =  24387.246117422153
Control only changes marginally.
RUN  1 , total integrated cost =  24387.246117422153
Improved over  1  iterations in  0.08550851047039032  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70170200134646 -56.70170337653204


ERROR:root:Problem in initial value trasfer


converged for  80
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.7500000000000004
-------  100 0.4500000000000001 0.7750000000000005
-------  105 0.5750000000000002 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  115 0.4250000000000001 0.8250000000000005
-------  120 0.5500000000000003 0.8250000000000005
-------  125 0.47500000000000014 0.8500000000000005
weight =  163.25056381702458
set cost params:  1.0 163.25056381702458 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  14459.407176661987
Gradient descend method:  None
RUN  1 , total integrated cost =  14459.407176661987
Control only changes marginally.
RUN  1 , total integrated cost =  14459.407176661987
Improved over  1  iterations in  0.10974347963929176  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67658665979109 -56.67660375812766
converged for  125
-------

In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

In [20]:
factor_iteration = 20
conv_0 = [[False]*2] * len(exc)
full_converge = False

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 20:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break
        
    counter += 1
    
    for i in i_range_0:
        print("------- ", i, exc[i], inh[i])
        
        if conv_0[i] == [True, True]:
            continue
            
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

    # exc and inh control current 

        setinit(initVars[i], aln)
        aln.params.duration = dur

        if not type(bestControl_0[i]) == type(None):
            control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
        else:
            control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
            weights_0[i] = weights_init[i]
            cost_0[i] = cost_init[i]

        cgv = None
        max_it = 500 * factor_iteration

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                           + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        with open(final_file,'wb') as f:
            pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                     costnode_0, weights_0], f)
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

--------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
weight =  11758.778823336454
set cost params:  1.0 11758.778823336454 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5901.90456448574
Gradient descend method:  None
RUN  1 , total integrated cost =  5542.327422624592
RUN  2 , total integrated cost =  4491.82819596091
RUN  3 , total integrated cost =  4353.670213264834
RUN  4 , total integrated cost =  4079.760765071986
RUN  5 , total integrated cost =  3939.407252428690

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  58 , total integrated cost =  3806.1338341094893
Improved over  58  iterations in  4.8010686580091715  seconds by  35.510074883070644  percent.
Problem in initial value trasfer:  Vmean_exc -56.6268293425474 -56.626833831176576
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
weight =  1762.8795138390672
set cost params:  1.0 1762.8795138390672 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5094.400010733049
Gradient descend method:  None
RUN  1 , total integrated cost =  4938.037728990757
RUN  2 , total integrated cost =  4191.222729473801
RUN  3 , total integrated cost =  4152.870572013158
RUN  4 , total integrated cost =  4103.813936860699
RUN  5 , total integrated cost =  4084.438127823503
RUN  6 , total integrated cost =  4047.1158162209117
RUN  7 , total integrated cost =  3898.7654778640776
RUN  8 , total integrated cost =  3881.4608305575484
RUN  9 , total integrated cost =  3851.1799601564812
R

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  218 , total integrated cost =  3266.763758765153
Improved over  218  iterations in  14.486952504143119  seconds by  35.875397458334106  percent.
Problem in initial value trasfer:  Vmean_exc -56.62449510169037 -56.62449317631398
no convergence
-------  10 0.4250000000000001 0.42500000000000016
weight =  2825.0554192590494
set cost params:  1.0 2825.0554192590494 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9108.232400397192
Gradient descend method:  None
RUN  1 , total integrated cost =  8432.05930224189
RUN  2 , total integrated cost =  6898.309655115094
RUN  3 , total integrated cost =  6828.795971048749
RUN  4 , total integrated cost =  6702.976311612386
RUN  5 , total integrated cost =  6660.879867544035
RUN  6 , total integrated cost =  6423.068206961809
RUN  7 , total integrated cost =  6174.714839189803
RUN  8 , total integrated cost =  6004.020386086318
RUN  9 , total integrated cost =  5949.115657289401
RUN  1

...

RUN  1 , total integrated cost =  15930.52476458639
Control only changes marginally.
RUN  1 , total integrated cost =  15930.52476458639
Improved over  1  iterations in  0.2304681409150362  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.68299326002809 -56.68300426362256
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.7500000000000004
-------  100 0.4500000000000001 0.7750000000000005
-------  105 0.5750000000000002 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
weight =  1249.9677155926474
set cost params:  1.0 1249.9677155926474 0.0
inter

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  19210.72933778856
Control only changes marginally.
RUN  1 , total integrated cost =  19210.72933778856
Improved over  1  iterations in  0.2472530249506235  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.693538332290565 -56.69351571816371
no convergence
-------  115 0.4250000000000001 0.8250000000000005
-------  120 0.5500000000000003 0.8250000000000005
weight =  4684.75663360979
set cost params:  1.0 4684.75663360979 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  28587.02429804899
Gradient descend method:  None
RUN  1 , total integrated cost =  28587.02429804899
Control only changes marginally.
RUN  1 , total integrated cost =  28587.02429804899
Improved over  1  iterations in  0.28892517648637295  seconds by  0.0  percent.
converged for  120
-------  125 0.47500000000000014 0.8500000000000005
-------  130 0.6000000000000003 0.8500000000000005
-------  135 0.5250000000000001 0.8750000000000006
--

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  19210.72933778856
Control only changes marginally.
RUN  1 , total integrated cost =  19210.72933778856
Improved over  1  iterations in  0.2241247482597828  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.693538332290565 -56.69351571816371
converged for  110
-------  115 0.4250000000000001 0.8250000000000005
-------  120 0.5500000000000003 0.8250000000000005
-------  125 0.47500000000000014 0.8500000000000005
-------  130 0.6000000000000003 0.8500000000000005
-------  135 0.5250000000000001 0.8750000000000006
-------  140 0.4500000000000001 0.9000000000000006
-------  145 0.5750000000000002 0.9000000000000006
--------------- 14
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, Tr

In [21]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [22]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

In [23]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    if counter > 20:
        break
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        
        if conv_1[i] == [True, True]:
            continue
            
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        if not type(bestControl_1[i]) == type(None):
            control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1].copy()
        else:
            control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1].copy()
            cost_1[i] = cost_0[i]
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(final_file_1,'wb') as f:
            pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)
            
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
    counter += 1

--------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  76.71287621631421
Gradient descend method:  None
RUN  1 , total integrated cost =  0.6420147443125521
RUN  2 , total integrated cost =  0.6401346152506305
RUN  3 , total integrated cost =  0.6282998695894045
RUN  4 , total integrated cost =  0.6106565007706124
RUN  5 , total integrated cost =  0.6091658541005933
RUN  6 , total integrated cost =  0.

ERROR:root:Problem in initial value trasfer


RUN  40 , total integrated cost =  0.37070350472151187
Control only changes marginally.
RUN  40 , total integrated cost =  0.37070350472151187
Improved over  40  iterations in  1.332348918542266  seconds by  99.51676495132811  percent.
Problem in initial value trasfer:  Vmean_exc -56.627621976793236 -56.62762179037296
no convergence
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  20.77105763106473
Gradient descend method:  None
RUN  1 , total integrated cost =  3.3682909362052236
RUN  2 , total integrated cost =  3.355974934263725
RUN  3 , total integrated cost =  3.355878052818582


ERROR:root:Problem in initial value trasfer


RUN  4 , total integrated cost =  3.355878052774776
RUN  5 , total integrated cost =  3.355878052774578
RUN  6 , total integrated cost =  3.3558780527745777
RUN  7 , total integrated cost =  3.3558780527745773
State only changes marginally.
RUN  8 , total integrated cost =  3.3558780527745773
Control only changes marginally.
RUN  8 , total integrated cost =  3.3558780527745773
Improved over  8  iterations in  0.2724869940429926  seconds by  83.8434897616595  percent.
Problem in initial value trasfer:  Vmean_exc -56.624563374891466 -56.624560746973565
no convergence
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  148.32476386447925
Gradient descend method:  None
RUN  1 , total integrated cost =  6.388337298813905
RUN  2 , total integrated cost =  6.3380954896203985
RUN  3 , total integrated cost =  6.336785668842365
RUN  4 , total integrated cost =  6.336370364714124
RUN  5 , total i

...

converged for  140
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  4.855892203425286
Gradient descend method:  None
RUN  1 , total integrated cost =  4.855892203425286
Control only changes marginally.
RUN  1 , total integrated cost =  4.855892203425286
Improved over  1  iterations in  0.06696634739637375  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354374741435 -56.70354360319175


ERROR:root:Problem in initial value trasfer


converged for  145
--------------- 3
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  18.014957739836362
Gradient descend method:  None
RUN  1 , total integrated cost =  18.014957739836362
Control only changes marginally.
RU

ERROR:root:Problem in initial value trasfer


converged for  25
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
set cost params:  1.0 1.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  11.676364353720707
Gradient descend method:  None
RUN  1 , total integrated cost =  11.676364353720707
Control only changes marginally.
RUN  1 , total integrated cost =  11.676364353720707
Improved over  1  iterations in  0.07465878501534462  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70174082377